In [2]:
import os
import seaborn as sns
import cv2
from tqdm.notebook import tqdm
import numpy as np
import random
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader
import time

In [3]:
import sys
sys.path.append('/content/drive/MyDrive/isic-project/src')

from image_utils import load_image, load_mask, normalize_image
from simple_unet import SimpleUNet

In [4]:
random.seed(42)
np.random.seed(42)

In [5]:
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [25]:
TRAIN_IMAGE_DIR = "/content/drive/MyDrive/isic-project/datasets/ISIC2018_Task1-2_Training_Input"
TRAIN_MASK_DIR = "/content/drive/MyDrive/isic-project/datasets/ISIC2018_Task1_Training_GroundTruth"

VAL_IMAGE_DIR = "/content/drive/MyDrive/isic-project/datasets/ISIC2018_Task1-2_Validation_Input"
VAL_MASK_DIR = "/content/drive/MyDrive/isic-project/datasets/ISIC2018_Task1_Validation_GroundTruth"

TARGET_SIZE = (256, 256)

In [27]:
def load_dataset(image_dir, mask_dir, num_samples=None):
    image_files = sorted(list(Path(image_dir).glob("*.jpg")))
    mask_files = sorted(list(Path(mask_dir).glob("*.png")))

    print(f"Найдено изображений: {len(image_files)}")
    print(f"Найдено масок: {len(mask_files)}")

    if num_samples is None:
      indices = range(len(image_files))
    else:
      indices = random.sample(range(len(image_files)), num_samples)

    images = []
    masks = []
    failed = 0

    for idx in tqdm(indices):
        try:
            img_path = image_files[idx]
            mask_path = mask_files[idx]
            image = load_image(img_path, TARGET_SIZE)
            mask = load_mask(mask_path, TARGET_SIZE)

            image_norm = normalize_image(image)

            images.append(image_norm)
            masks.append(mask)
        except Exception as e:
            failed += 1

    print(f"Успешно загружено: {len(images)} пар")
    print(f"Не удалось загрузить: {failed}")

    print(f"\nФорма изображений: {np.array(images).shape}")
    print(f"Форма масок: {np.array(masks).shape}")

    return images, masks


In [ ]:
train_images, train_masks = load_dataset(TRAIN_IMAGE_DIR, TRAIN_MASK_DIR)

Найдено изображений: 2594
Найдено масок: 2594


  0%|          | 0/500 [00:00<?, ?it/s]

In [28]:
val_images, val_masks = load_dataset(VAL_IMAGE_DIR, VAL_MASK_DIR)

Найдено изображений: 100
Найдено масок: 100


  0%|          | 0/100 [00:00<?, ?it/s]

Успешно загружено: 100 пар
Не удалось загрузить: 0

Форма изображений: (100, 256, 256, 3)
Форма масок: (100, 256, 256)


In [17]:
def plot_sample(image, mask):
    """Визуализация примера"""
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    axes[0].imshow(image)
    axes[0].set_title("Изображение")
    axes[0].axis('off')

    axes[1].imshow(mask, cmap='gray')
    axes[1].set_title("Маска")
    axes[1].axis('off')

    axes[2].imshow(image)
    axes[2].imshow(mask, cmap='jet', alpha=0.5)
    axes[2].set_title("Наложение")
    axes[2].axis('off')

    plt.tight_layout()
    return fig

In [ ]:
sample_idx = random.choice(range(len(train_images)))
plot_sample(train_images[sample_idx], train_masks[sample_idx])
plt.show()

In [ ]:
sample_idx = random.choice(range(len(val_images)))
plot_sample(val_images[sample_idx], val_masks[sample_idx])
plt.show()

In [ ]:
lesion_percentages = []
for mask in train_masks:
    lesion_pixels = np.sum(mask > 0.5)
    total_pixels = mask.size
    lesion_percentages.append(lesion_pixels / total_pixels * 100)

lesion_percentages = np.array(lesion_percentages)

percentiles = [5, 25, 50, 75, 95]

plt.figure(figsize=(8, 4))
sns.histplot(lesion_percentages, bins=30, kde=True, color='purple')
plt.title('Распределение процента поражения')
plt.xlabel('Процент поражения')
plt.ylabel('Количество изображений')
plt.grid(True, alpha=0.3)

colors = ['red', 'orange', 'green', 'blue', 'purple', 'black']
for p, color in zip(percentiles, colors):
    value = np.percentile(lesion_percentages, p)
    plt.axvline(value, color=color, linestyle='--', alpha=0.7, label=f'{p}%: {value:.1f}%')

plt.legend()
plt.tight_layout()
plt.show()

In [18]:
class SegmentationDataset:
    def __init__(self, images, masks, augment=False):
        self.images = images
        self.masks = masks
        self.augment = augment

    def __len__(self):
        return len(self.images)

    def _apply_augmentation(self, image, mask):
        # Горизонтальное отражение (50%)
        if np.random.rand() > 0.5:
            image = image[:, ::-1, :]
            mask = mask[:, ::-1]

        # Вертикальное отражение (50%)
        if np.random.rand() > 0.5:
            image = image[::-1, :, :]
            mask = mask[::-1, :]

        # Повороты
        if np.random.rand() > 0.5:
            k = np.random.randint(0, 4)
            if k > 0:
                image = np.rot90(image, k)
                mask = np.rot90(mask, k)

        return image.copy(), mask.copy()

    def __getitem__(self, idx):
        image = self.images[idx].astype(np.float32)
        mask = self.masks[idx].astype(np.float32)

        if self.augment:
            image, mask = self._apply_augmentation(image, mask)

        image_tensor = torch.from_numpy(image).permute(2, 0, 1).float()
        mask_tensor = torch.from_numpy(mask).unsqueeze(0).float()

        return image_tensor, mask_tensor

In [ ]:
BATCH_SIZE = 4

train_dataset = SegmentationDataset(train_images, train_masks, augment=True)
val_dataset = SegmentationDataset(val_images, val_masks)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train loader: {len(train_loader)} батчей по {BATCH_SIZE}")
print(f"Val loader: {len(val_loader)} батчей по {BATCH_SIZE}")

Train loader: 100 батчей по 4
Val loader: 25 батчей по 4


In [10]:
device = torch.device('cuda')

In [6]:
# SimpleUNet is now imported from src/simple_unet.py
# No need to duplicate the model definition here


In [ ]:
model = SimpleUNet()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

print(f"Параметров модели: {sum(p.numel() for p in model.parameters()):,}")

Параметров модели: 31,043,521


In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    start_time = time.time()
    model.train()
    epoch_loss = 0

    for _, (images, masks) in enumerate(loader):
        images = images.to(device)
        masks = masks.to(device)

        predictions = model(images)
        loss = criterion(predictions, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    epoch_time = time.time() - start_time
    avg_loss = epoch_loss / len(loader)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}, Time: {epoch_time:.1f}s")
    return avg_loss

In [ ]:
def calculate_dice_score(pred, target, threshold=0.5):
    pred_binary = (pred > threshold).float()

    pred_flat = pred_binary.view(-1)
    target_flat = target.view(-1)

    intersection = (pred_flat * target_flat).sum()
    dice = (2. * intersection) / (pred_flat.sum() + target_flat.sum() + 1e-6)

    return dice.item()

def calculate_iou(pred, target, threshold=0.5):
    pred_binary = (pred > threshold).float()

    pred_flat = pred_binary.view(-1)
    target_flat = target.view(-1)

    intersection = (pred_flat * target_flat).sum()
    union = pred_flat.sum() + target_flat.sum() - intersection

    return (intersection / (union + 1e-6)).item()


In [ ]:
def validate(model, loader, criterion, device):
    model.eval()
    val_loss = 0
    val_dice = 0
    val_iou = 0

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)

            predictions = model(images)
            loss = criterion(predictions, masks)

            dice = calculate_dice_score(predictions, masks)
            iou = calculate_iou(predictions, masks)

            val_loss += loss.item()
            val_dice += dice
            val_iou += iou

    return (val_loss / len(loader),
            val_dice / len(loader),
            val_iou / len(loader))

In [8]:
BEST_MODEL_PATH = '/content/drive/MyDrive/isic-project/models/segmentation_model.pth'

In [ ]:
NUM_EPOCHS = 30

model.to(device)

history = {
    'train_loss': [],
    'val_loss': [],
    'val_dice': [],
    'val_iou': []
}

for epoch in range(NUM_EPOCHS):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)

    val_loss, val_dice, val_iou = validate(model, val_loader, criterion, device)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_dice'].append(val_dice)
    history['val_iou'].append(val_iou)

    if epoch == 0 or val_dice > max(history['val_dice'][:-1]):
        print("Лучший результат:")
        print({
            'epoch': epoch,
            'val_dice': val_dice,
            'val_iou': val_iou,
        })

        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_dice': val_dice,
            'val_iou': val_iou,
        }, BEST_MODEL_PATH)

Epoch 1, Loss: 0.1664, Time: 25.5s
Лучший результат:
{'epoch': 0, 'val_dice': 0.8375943279266358, 'val_iou': 0.7299941301345825}
Epoch 2, Loss: 0.1847, Time: 24.6s
Epoch 3, Loss: 0.1593, Time: 24.5s
Лучший результат:
{'epoch': 2, 'val_dice': 0.8525335693359375, 'val_iou': 0.7534410643577576}
Epoch 4, Loss: 0.1647, Time: 24.8s
Лучший результат:
{'epoch': 3, 'val_dice': 0.8634047317504883, 'val_iou': 0.766505835056305}
Epoch 5, Loss: 0.1568, Time: 24.7s
Epoch 6, Loss: 0.1455, Time: 24.6s
Epoch 7, Loss: 0.1589, Time: 24.6s
Лучший результат:
{'epoch': 6, 'val_dice': 0.8731563782691956, 'val_iou': 0.7826053547859192}
Epoch 8, Loss: 0.1596, Time: 24.9s
Epoch 9, Loss: 0.1655, Time: 24.7s
Epoch 10, Loss: 0.1508, Time: 24.6s
Epoch 11, Loss: 0.1494, Time: 24.6s
Epoch 12, Loss: 0.1610, Time: 24.6s
Epoch 13, Loss: 0.1590, Time: 24.6s
Epoch 14, Loss: 0.1445, Time: 24.6s
Epoch 15, Loss: 0.1466, Time: 24.6s
Epoch 16, Loss: 0.1500, Time: 24.5s
Epoch 17, Loss: 0.1463, Time: 24.5s
Epoch 18, Loss: 0.1505

In [15]:
def visualize_prediction(model, image, mask, device):
    input_tensor = image.unsqueeze(0).to(device)

    with torch.no_grad():
        prediction = model(input_tensor)
        prediction_np = prediction[0, 0].cpu().numpy()

    # Подготавливаем данные для отображения
    image_np = image.permute(1, 2, 0).numpy()
    mask_np = mask[0].numpy()
    binary_mask = (prediction_np > 0.5).astype(np.float32)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))

    axes[0].imshow(image_np)
    axes[0].set_title(f"Изображение {idx}")
    axes[0].axis('off')

    axes[1].imshow(mask_np, cmap='gray')
    axes[1].set_title("Истинная маска")
    axes[1].axis('off')

    axes[2].imshow(prediction_np, cmap='gray')
    axes[2].set_title("Предсказания")
    axes[2].axis('off')

    axes[3].imshow(binary_mask, cmap='gray')
    axes[3].set_title("Бинарная маска (>0.5)")
    axes[3].axis('off')

    dice = calculate_dice_score(
        torch.tensor(prediction_np).unsqueeze(0).unsqueeze(0),
        torch.tensor(mask_np).unsqueeze(0).unsqueeze(0)
    )

    plt.suptitle(f"Индекс: {idx}, Dice: {dice:.3f}", y=1.05)
    plt.tight_layout()
    plt.show()

In [11]:
loaded_model = SimpleUNet()

checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)

loaded_model.load_state_dict(checkpoint['model_state_dict'])
loaded_model.to(device)
loaded_model.eval()

print(f"Параметров модели: {sum(p.numel() for p in loaded_model.parameters()):,}")

Параметров модели: 31,043,521


In [16]:
idx = np.random.randint(len(val_dataset))
image, mask = val_dataset[idx]

visualize_prediction(loaded_model, image, mask, device)

NameError: name 'val_dataset' is not defined